# Read in the split data and load the YAML metadata

In [ ]:
# Import the required modules

import pandas as pd
pd.set_option('display.float_format', lambda x: '%.3f' % x)

import numpy as np
import scipy as sp

import yaml
import matplotlib.pyplot as plt

from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics.pairwise import pairwise_distances_argmin
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.linear_model import Lasso
import numpy as np
from sklearn.experimental import enable_halving_search_cv

In [ ]:
# Import the split data
X_train_path = 'bin/X_train.csv'
y_train_path = 'bin/y_train.csv'
X_test_path = 'bin/X_test.csv'
y_test_path = 'bin/y_test.csv'

X_train = pd.read_csv(X_train_path)
y_train = pd.read_csv(y_train_path)
X_test = pd.read_csv(X_test_path)
y_test = pd.read_csv(y_test_path)

# flatten target 
target_variable = 'Beta_winsorized'
y_train = y_train[target_variable].values.ravel()
y_test  = y_test[target_variable].values.ravel()

# print the shape of the data
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

# print the first 5 rows of the data
print(X_train[:5])
print(y_train[:5])
print(X_test[:5])
print(y_test[:5])
	


In [ ]:
# ---- metadata needed for feature-name lookups ----
import yaml

# variable labels (short human-readable descriptions)
with open('variable_labels.yaml') as f:
    variable_labels_dict = yaml.safe_load(f)

# SAS format for each original variable (tells you if cat / numeric)
with open('variable_formats.yaml') as f:
    variable_formats_dict = yaml.safe_load(f)

# mapping from format-name → {category-code: category-text}
with open('categorical_variables_categories.yaml') as f:
    categorical_variables_categories_dict = yaml.safe_load(f)

# list of all one-hot (dummy) columns produced during cleaning
with open('cleaned_variables.yaml') as f:
    cleaned_vars = yaml.safe_load(f)
new_categorical_variables = set(cleaned_vars['categorical_variables'])

In [ ]:
# Set the target variable
target_variable = 'Beta_winsorized'

# print the value of the target variable from the training data to validate it loaded correctly
print(pd.Series(y_train, name=target_variable).head())

# Supervised Learning Models: Linear, LASSO, and Random Forest, Gradient Boosting,

### Helper Function

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold
from sklearn.experimental import enable_halving_search_cv   # noqa: F401
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np


def tune_and_evaluate(
        model_cls,                # e.g. HistGradientBoostingRegressor
        param_grid,               # dict of hyper-parameters
        X_train, y_train,
        X_test, y_test,
        metric='neg_root_mean_squared_error',
        cv_splits=5,
        random_state=42,
        label='',
        search_strategy='grid',   # 'grid' | 'random' | 'halving'
        n_iter=10,                # for RandomizedSearchCV
        factor=3                  # for HalvingGridSearchCV
):
    """Hyper-parameter tuning + evaluation wrapper."""

    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    # Instantiate base model, passing random_state only if supported
    base_kwargs = {'random_state': random_state} if 'random_state' in model_cls().get_params() else {}
    base = model_cls(**base_kwargs)

    # Map strategy → (CV class, kwargs)
    strategy_map = {
        'grid':    (GridSearchCV,         {'param_grid': param_grid}),
        'random':  (RandomizedSearchCV,   {'param_distributions': param_grid,
                                           'n_iter': n_iter,
                                           'random_state': random_state}),
        'halving': (HalvingGridSearchCV,  {'param_grid': param_grid,
                                           'factor': factor,
                                           'random_state': random_state})
    }

    try:
        search_class, extra_kwargs = strategy_map[search_strategy]
    except KeyError:
        raise ValueError(f"Invalid search_strategy '{search_strategy}'. "
                         f"Choose from {list(strategy_map)}")

    search = search_class(
        estimator=base,
        scoring=metric,
        cv=cv,
        n_jobs=-1,
        verbose=1,
        **extra_kwargs
    ).fit(X_train, y_train)

    best_model = search.best_estimator_

    y_pred = best_model.predict(X_test)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_pred)

    result = {
        'model': f'{label or model_cls.__name__} (tuned - {search_strategy.capitalize()})',
        'MAE':   mae,
        'RMSE':  rmse,
        'best_params': search.best_params_,
        'best_score':  search.best_score_
    }
    return best_model, result

### Fit a Histogram Gradient Boosting Model


In [ ]:
gb_grid = {
    'learning_rate': [0.03, 0.04, 0.05, 0.06] ,
    'max_depth':     [12, 15, 18] ,
    'max_iter':      [300, 400, 600],
    'l2_regularization': [0, 0.0005, 0.001, 0.01],
    'min_samples_leaf': [10, 20],
    'max_leaf_nodes': [31, 63]
}

best_gb, gb_metrics = tune_and_evaluate(
    model_cls=HistGradientBoostingRegressor,
    param_grid=gb_grid,
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test,
    label='Gradient Boosting',
    search_strategy='halving', 
    factor=3                   
)


print(gb_metrics)

### Fit a Linear Regression Model

* Fit a Linear Model on the training partition, and evaluate it on the testing partition
* There is a lot of Multicollinearity in this model because we are putting all predictor variables into the model
* https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [ ]:
linreg = LinearRegression()
linreg.fit(X_train, y_train)

mae  = mean_absolute_error(y_test, linreg.predict(X_test))
mse  = mean_squared_error(y_test, linreg.predict(X_test))   
rmse = np.sqrt(mse)

linreg_metrics = {
    'model': 'Linear Regression',
    'MAE':   mae,
    'RMSE':  rmse
}

print(linreg_metrics)

### Fit a LASSO Model 

* This will shrink most of the variable coefficients to zero for automated variable selection
* https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso


In [ ]:
lasso_grid = {
    'alpha': np.logspace(-4, -2, 9),   # 0.001 … 1.0
    'max_iter': [5000],
    'tol': [1e-4]
}

best_lasso, lasso_metrics = tune_and_evaluate(
    model_cls=Lasso,
    param_grid=lasso_grid,
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test,
    label='Lasso'
)

print(lasso_metrics)

### Fit a Random Forest Regression Model

* https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

In [9]:
rf_grid = {
    'n_estimators':     [400, 600, 800],
    'max_depth':        [18, 22, 26],
    'min_samples_leaf':[1, 2, 4] ,
    'max_features':    ['sqrt'],
    'min_samples_split': [2, 5, 10] # Corrected 'min_samples_split' key
}

best_rf, rf_metrics = tune_and_evaluate(
    model_cls=RandomForestRegressor,
    param_grid=rf_grid,
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test,
    label='Random Forest',
    search_strategy='halving',
    factor=3
)
print(rf_metrics)

KeyboardInterrupt: 

In [ ]:
all_metrics = [gb_metrics, linreg_metrics, lasso_metrics, rf_metrics]
pd.DataFrame(all_metrics)

### Use the models to identify the most useful variable to predict the target

In [ ]:
# === Linear Regression: largest absolute coefficients ================

TOP_N = 20
lin_coef = pd.Series(linreg.coef_, index=X_train.columns)
lin_top = lin_coef.abs().sort_values(ascending=False).head(TOP_N)

print(f"Top {TOP_N} Linear-Regression coefficients")
for var, coef in lin_top.items():
    sign = "+" if coef > 0 else "-"
    if var in new_categorical_variables:
        parts = var.split("_")
        cat_var, category = "_".join(parts[:-1]), parts[-1]
        category = int(category) if category.isdigit() else category
        label = categorical_variables_categories_dict[variable_formats_dict[cat_var]][category]
        print(f"{var} | {variable_labels_dict[cat_var]} = {label} | coef {sign}{abs(coef):.4f}")
    else:
        print(f"{var} | {variable_labels_dict[var]} | coef {sign}{abs(coef):.4f}")

In [ ]:
# === HistGradientBoosting – top N features by permutation importance ==
from sklearn.inspection import permutation_importance
import pandas as pd
import numpy as np

TOP_N = 20

perm = permutation_importance(
    estimator=best_gb,                 # fitted model
    X=X_test, y=y_test,                # use hold-out set
    n_repeats=10,
    random_state=0,
    n_jobs=-1
)

gb_importances = pd.Series(perm.importances_mean, index=X_train.columns)
gb_top = gb_importances.abs().sort_values(ascending=False).head(TOP_N)

print(f"Top {TOP_N} Gradient-Boosting features (permutation importance)")
for var, imp in gb_top.items():
    if var in new_categorical_variables:
        parts = var.split("_")
        cat_var, category = "_".join(parts[:-1]), parts[-1]
        category = int(category) if category.isdigit() else category
        label = categorical_variables_categories_dict[variable_formats_dict[cat_var]][category]
        print(f"{var} | {variable_labels_dict[cat_var]} = {label} | ΔRMSE {imp:.4f}")
    else:
        print(f"{var} | {variable_labels_dict[var]} | ΔRMSE {imp:.4f}")

In [ ]:
# === Lasso: non-zero coefficients (already sparse) ====================

lasso_coef = pd.Series(best_lasso.coef_, index=X_train.columns)
nz = lasso_coef[lasso_coef != 0].sort_values(key=np.abs, ascending=False)

print(f"{len(nz)} non-zero Lasso coefficients")
for var, coef in nz.items():
    sign = "+" if coef > 0 else "-"
    if var in new_categorical_variables:
        parts = var.split("_")
        cat_var, category = "_".join(parts[:-1]), parts[-1]
        category = int(category) if category.isdigit() else category
        label = categorical_variables_categories_dict[variable_formats_dict[cat_var]][category]
        print(f"{var} | {variable_labels_dict[cat_var]} = {label} | coef {sign}{abs(coef):.4f}")
    else:
        print(f"{var} | {variable_labels_dict[var]} | coef {sign}{abs(coef):.4f}")

In [ ]:
# === Random Forest: top N features by importance ======================

TOP_N = 20
rf_importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)
rf_top = rf_importances.sort_values(ascending=False).head(TOP_N)

print(f"Top {TOP_N} Random-Forest features")
for var, imp in rf_top.items():
    if var in new_categorical_variables:
        parts = var.split("_")
        cat_var, category = "_".join(parts[:-1]), parts[-1]
        category = int(category) if category.isdigit() else category
        label = categorical_variables_categories_dict[variable_formats_dict[cat_var]][category]
        print(f"{var} | {variable_labels_dict[cat_var]} = {label} | importance {imp:.4f}")
    else:
        print(f"{var} | {variable_labels_dict[var]} | importance {imp:.4f}")

In [ ]:
# --- Compare all supervised models -----------------------------------
import pandas as pd

all_metrics = [gb_metrics, linreg_metrics, lasso_metrics, rf_metrics]

results_df = (pd.DataFrame(all_metrics)
                .set_index('model')
                .sort_values('RMSE'))   # sort best → worst by RMSE

display(results_df.style.format({'MAE': '{:.4f}', 'RMSE': '{:.4f}'}))

In [ ]:
results_df[['RMSE']].plot(kind='barh', figsize=(6,3), legend=False,
                          title='Test RMSE by Model').invert_yaxis()
plt.xlabel('RMSE')
plt.show()

## Unsupervised Learning

In [ ]:
# 1) Decide the source matrix for clustering
#    – If you want clusters on the *full* data (train + test), concatenate:
X_cluster = pd.concat([X_train, X_test], ignore_index=True)

# 2) Optionally: keep only numeric columns.  One-hot dummies are fine,
#    but if you have explicit 0/1 flags you may prefer to leave them as ints.
num_cols = X_cluster.select_dtypes(include=[np.number]).columns
X_cluster_numeric = X_cluster[num_cols]

# 3) Scale *just for clustering*
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster_numeric)

print(f"Original shape: {X_cluster_numeric.shape},  scaled shape: {X_scaled.shape}")

In [ ]:
# --- Mini-Batch K-means elbow plot -----------------------------------
from sklearn.cluster import MiniBatchKMeans
import matplotlib.pyplot as plt

inertias = []

for k in range(1, 11):
    mbk = MiniBatchKMeans(
        n_clusters=k,
        init='k-means++',
        batch_size=100,
        n_init=10,
        max_no_improvement=10,
        random_state=0,
        verbose=0
    )
    mbk.fit(X_scaled)               
    inertias.append(mbk.inertia_)

plt.plot(range(1, 11), inertias, marker='o')
plt.xticks(range(1, 11))
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Mini-Batch K-Means')
plt.show()

In [ ]:
# Plot of the Number of Clusters vs. the Inertia of Mini-Batch K-means
plt.plot(range(1, 11), inertias)        
plt.xticks(range(1, 11))
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (sum of squared distances)")
plt.title("Elbow Method for Mini-Batch K-means")
plt.show()

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import pairwise_distances_argmin

# Choose the matrix for clustering
X_kmeans = X_scaled          # or X_pca

k = 3                        # elbow choice
mbk = MiniBatchKMeans(init='k-means++',
                      n_clusters=k,
                      batch_size=100,
                      n_init=10,
                      max_no_improvement=10,
                      random_state=0,
                      verbose=0)

mbk.fit(X_kmeans)

cluster_centers = np.sort(mbk.cluster_centers_, axis=0)
labels = pairwise_distances_argmin(X_kmeans, cluster_centers)

print(labels)                # cluster assignment per observation

In [ ]:
# --- PCA --------------------------------------------------------------
import pandas as pd, numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

print("Starting PCA…")

# Use the scaled matrix you built earlier
X_for_pca = X_scaled           # or X_cluster_numeric, or X_kmeans – pick the right one

pca = PCA(n_components=0.95, random_state=0)
X_pca = pca.fit_transform(X_for_pca)

print(f"Number of PCs selected: {pca.n_components_}")

cum_var = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, pca.n_components_+1), pca.explained_variance_ratio_, 'o--', label='Individual')
plt.plot(range(1, pca.n_components_+1), cum_var, 'o-', label='Cumulative')
plt.xlabel('Principal component'); plt.ylabel('Variance explained')
plt.title('Scree plot'); plt.grid(True); plt.legend(); plt.show()

print("Shape after PCA:", X_pca.shape)
X_pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
display(X_pca_df.head())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt

# Assuming 'X_scaled' is your scaled feature matrix.
# Or, you might want to use 'X_pca' from the PCA step above.
# For demonstration, let's create a dummy X_scaled:
# X_scaled = pd.DataFrame(np.random.rand(100, 10)) # Replace with your actual X_scaled or X_pca

# --- Start of GMM ---
print("Starting GMM...")

# Define the range of components (clusters) to test
n_components_range = range(1, 11) # Example: 1 to 10 components
covariance_types = ['spherical', 'tied', 'diag', 'full'] # Common covariance types

# We'll use 'full' covariance for this example, but you should test others
# and select the best based on BIC/AIC or other criteria.
# For a fixed number of components (e.g., 4, to match your K-means)
n_clusters_gmm = 4 # Example, tune this parameter
gmm = GaussianMixture(n_components=n_clusters_gmm, covariance_type='full', random_state=42)

# Fit GMM
# If using PCA-transformed data: gmm.fit(X_pca)
gmm.fit(X_scaled)

# Get cluster assignments
gmm_labels = gmm.predict(X_scaled) # or X_pca
# Get probabilities of belonging to each cluster
gmm_probs = gmm.predict_proba(X_scaled) # or X_pca

print(f"\nCluster labels for the first 10 data points: {gmm_labels[:10]}")
# print(f"Probabilities for the first data point: {gmm_probs[0]}")
print(f"Number of clusters found: {len(np.unique(gmm_labels))}")

# To find the optimal number of components, you can iterate and check BIC/AIC
bics = []
aics = []

print("\nCalculating BIC and AIC for different numbers of components...")
for n_components in n_components_range:
    gmm_iter = GaussianMixture(n_components=n_components, covariance_type='full', random_state=42)
    gmm_iter.fit(X_scaled) # or X_pca
    bics.append(gmm_iter.bic(X_scaled)) # or X_pca
    aics.append(gmm_iter.aic(X_scaled)) # or X_pca

# Plot BIC and AIC
plt.figure(figsize=(10, 5))
plt.plot(n_components_range, bics, marker='o', label='BIC')
plt.plot(n_components_range, aics, marker='o', label='AIC')
plt.title('GMM: BIC and AIC for Number of Components')
plt.xlabel('Number of Components')
plt.ylabel('Information Criterion Value (Lower is Better)')
plt.xticks(n_components_range)
plt.legend()
plt.grid(True)
plt.show()

# The optimal number of components is often where BIC/AIC is minimized.
# --- End of GMM ---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors # For helping to choose eps
import matplotlib.pyplot as plt



print("Starting DBSCAN...")


min_samples_dbscan = 10 # Tune this parameter

print(f"Calculating k-distances for eps estimation (min_samples = {min_samples_dbscan})...")

neigh = NearestNeighbors(n_neighbors=min_samples_dbscan)
nbrs = neigh.fit(X_scaled) # or X_pca
distances, indices = nbrs.kneighbors(X_scaled) # or X_pca

# Sort the distances to the k-th nearest neighbor
k_distances = np.sort(distances[:, min_samples_dbscan-1], axis=0)

# Plot the k-distance graph
plt.figure(figsize=(10, 6))
plt.plot(k_distances)
plt.title(f'K-distance Graph (k={min_samples_dbscan}) for DBSCAN eps Estimation')
plt.xlabel('Points sorted by distance to k-th nearest neighbor')
plt.ylabel(f'{min_samples_dbscan}-th Nearest Neighbor Distance (eps candidate)')
plt.grid(True)
# Look for the "elbow" or "knee" in this plot to choose eps.
# This point indicates a region where distances start to increase sharply.
plt.show()
print("Review the K-distance plot above to choose a suitable 'eps' value (y-axis value at the 'elbow').")


# --- DBSCAN Clustering ---
# You need to set 'eps' based on the k-distance plot.
# Let's pick an example value for eps; YOU MUST TUNE THIS.
eps_dbscan = 0.5 # Example value, TUNE THIS CAREFULLY based on the plot above.

print(f"\nRunning DBSCAN with eps={eps_dbscan} and min_samples={min_samples_dbscan}...")
dbscan = DBSCAN(eps=eps_dbscan, min_samples=min_samples_dbscan)

# Fit DBSCAN
# If using PCA-transformed data: dbscan.fit(X_pca)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Number of clusters in labels, ignoring noise if present.
# Noise points are labeled as -1.
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_dbscan = list(dbscan_labels).count(-1)

print(f"\nEstimated number of clusters: {n_clusters_dbscan}")
print(f"Estimated number of noise points: {n_noise_dbscan}")
# print(f"Cluster labels: {np.unique(dbscan_labels)}") # Shows all unique labels including -1 for noise

# --- End of DBSCAN ---

## Business Question

**Can we predict the systematic risk (Beta) of mortgages in the FHFA NSMO portfolio using mortgage characteristics and macroeconomic factors?**

### Supervised Learning Questions

- **Which mortgage features (loan characteristics, borrower profiles, property details) are most predictive of mortgage Beta?**
- **How accurately can different ML models predict mortgage Beta?**
- **Which model family (probabilistic, tree-based, instance-based) performs best for this financial prediction task?**

### Unsupervised Learning Questions

- **Are there natural groupings/clusters of mortgages based on their characteristics?**
- **Do these clusters correspond to different risk profiles (Beta levels)?**
- **Can we identify mortgage segments that behave similarly in terms of systematic risk?**

### Financial/Business Questions

- **How do mortgage characteristics influence systematic risk relative to market movements?**
- **Can we identify high-Beta vs low-Beta mortgage profiles for portfolio management?**